In [1]:
from google.colab import drive
drive.mount('/content/drive')

#ini bagian buat folder project di drive
import os

PROJECT_ROOT = "/content/drive/MyDrive/skin-xai-skripsi"
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/data/raw", exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/data/processed", exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/outputs/models", exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/outputs/images", exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/experiments", exist_ok=True)

print("yyeeeyyy struktur foldernya udah dibuat:")
for root, dirs, files in os.walk(PROJECT_ROOT):
  level = root.replace(PROJECT_ROOT, '').count(os.sep)
  indent = ' ' * 2 * (level)
  print(f'{indent}{os.path.basename(root)}/')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
yyeeeyyy struktur foldernya udah dibuat:
skin-xai-skripsi/
  data/
    raw/
    processed/
  outputs/
    models/
    images/
  experiments/


In [2]:
#ini bagian install library
!pip install -q roboflow

#test versi
import tensorflow as tf
import numpy as np
import matplotlib
import sklearn
import cv2

print(f"TensorFlow  : {tf.__version__}")
print(f"NumPy       : {np.__version__}")
print(f"Matplotlib  : {matplotlib.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")
print(f"OpenCV      : {cv2.__version__}")
print(f"GPU tersedia: {tf.config.list_physical_devices('GPU')}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 126.0 MB/s eta 0:00:00
TensorFlow  : 2.20.0
NumPy       : 2.0.2
Matplotlib  : 3.10.0
Scikit-learn: 1.6.1
OpenCV      : 4.10.0
GPU tersedia: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [16]:
from google.colab import userdata

# Set git identity
!git config --global user.email "sukarami.1996@gmail.com "
!git config --global user.name "srohimatuzz"

# Clone repo
REPO_URL = "https://github.com/srohimatuzz/skin-xai-skripsi.git"
REPO_DIR = "/content/skin-xai-skripsi"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
    print("Repo berhasil di-clone")
else:
    print("Repo sudah ada, pull latest changes")
    !cd {REPO_DIR} && git pull

Repo sudah ada, pull latest changes
Already up to date.


In [5]:
from roboflow import Roboflow
from google.colab import userdata
import os

api_key = userdata.get('ROBOFLOW_API_KEY')
rf = Roboflow(api_key=api_key)

print("suksezzzz, roboflow udah terconnect")
print(f" PROJECT_ROOT : {PROJECT_ROOT}")

suksezzzz, roboflow udah terconnect
 PROJECT_ROOT : /content/drive/MyDrive/skin-xai-skripsi


In [7]:
#download dataset 1 Tri-Dermatosis
print("prosess download dataset 1 tri-dermatosis...")
print("ditunggu dulu yaawww....\n")

project1 = rf.workspace("umie-fatihah").project("tri-dermatosis-skin-detection-3")
dataset1 = project1.version(4).download(
    model_format="folder",
    location=f"{PROJECT_ROOT}/data/raw/dataset1",
    overwrite=False
    )

print("\n dataset 1 doneeee")
print(f"lokasinya: {PROJECT_ROOT}/data/raw/dataset1")

prosess download dataset 1 tri-dermatosis...
ditunggu dulu yaawww....

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/drive/MyDrive/skin-xai-skripsi/data/raw/dataset1 in folder:: 100%|██████████| 1108/1108 [00:11<00:00, 95.68it/s] 


 dataset 1 doneeee
lokasinya: /content/drive/MyDrive/skin-xai-skripsi/data/raw/dataset1


In [8]:
# DOWNLOAD DATASET 2 — Darmatological Diagnosis
print("prosess download dataset 2 darmatological diagnosis...")
print("ditunggu dulu yaawww....\n")

project2 = rf.workspace("diseaseanotation").project("darmatological-diagnosis")
dataset2 = project2.version(4).download(
    model_format="folder",
    location=f"{PROJECT_ROOT}/data/raw/dataset2",
    overwrite=False
    )

print("\n dataset 2 doneeee")
print(f"lokasinya: {PROJECT_ROOT}/data/raw/dataset2")

prosess download dataset 2 darmatological diagnosis...
ditunggu dulu yaawww....

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/drive/MyDrive/skin-xai-skripsi/data/raw/dataset2 in folder:: 100%|██████████| 1512/1512 [00:15<00:00, 98.52it/s] 


 dataset 2 doneeee
lokasinya: /content/drive/MyDrive/skin-xai-skripsi/data/raw/dataset2


In [10]:
# ============================================
# VERIFIKASI — Cek semua kelas yang ter-download
# ============================================

def audit_dataset(base_path, dataset_name):
    """Audit lengkap struktur dan jumlah gambar dataset."""
    print(f"\n{'='*55}")
    print(f"  {dataset_name}")
    print(f"{'='*55}")

    if not os.path.exists(base_path):
        print(f"=======Path tidak ditemukan: {base_path}")
        return {}

    total_all = 0
    summary = {}

    for split in ["train", "valid", "test"]:
        split_path = os.path.join(base_path, split)
        if not os.path.exists(split_path):
            print(f"  !!!!  Split '{split}' tidak ditemukan")
            continue

        print(f"\n  [{split.upper()}]")
        summary[split] = {}

        classes = sorted(os.listdir(split_path))
        for cls in classes:
            cls_path = os.path.join(split_path, cls)
            if not os.path.isdir(cls_path):
                continue
            count = len([
                f for f in os.listdir(cls_path)
                if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))
            ])
            summary[split][cls] = count
            total_all += count

            # Tandai kelas yang bukan target
            is_target = any(k in cls.lower() for k in ['eczema', 'psoriasis'])
            flag = "✅" if is_target else "[AKAN DIFILTER]"
            print(f"    {flag} {cls:30s}: {count:4d} gambar")

    print(f"\n  TOTAL semua kelas : {total_all} gambar")
    return summary

# Jalankan audit kedua dataset
d1_summary = audit_dataset(
    f"{PROJECT_ROOT}/data/raw/dataset1",
    "DATASET 1 — Tri-Dermatosis (Umie Fatihah)"
)

d2_summary = audit_dataset(
    f"{PROJECT_ROOT}/data/raw/dataset2",
    "DATASET 2 — Darmatological Diagnosis"
)

print("\n" + "="*55)
print("  RINGKASAN KELAS TARGET (Eczema + Psoriasis)")
print("="*55)

for ds_name, summary in [("Dataset 1", d1_summary), ("Dataset 2", d2_summary)]:
    print(f"\n{ds_name}:")
    for split, classes in summary.items():
        target = {k: v for k, v in classes.items()
                  if any(t in k.lower() for t in ['eczema', 'psoriasis'])}
        if target:
            total = sum(target.values())
            print(f"  {split:6s}: {target} → subtotal {total}")


  DATASET 1 — Tri-Dermatosis (Umie Fatihah)

  [TRAIN]
    ✅ eczema                        :  298 gambar
    ✅ psoriasis                     :  299 gambar
    [AKAN DIFILTER] scabies                       :  160 gambar

  [VALID]
    ✅ eczema                        :   88 gambar
    ✅ psoriasis                     :   90 gambar
    [AKAN DIFILTER] scabies                       :   46 gambar

  [TEST]
    ✅ eczema                        :   44 gambar
    ✅ psoriasis                     :   46 gambar
    [AKAN DIFILTER] scabies                       :   23 gambar

  TOTAL semua kelas : 1094 gambar

  DATASET 2 — Darmatological Diagnosis

  [TRAIN]
    [AKAN DIFILTER] Acne                          :  114 gambar
    [AKAN DIFILTER] Acne Actinic Keratosis        :    1 gambar
    ✅ Acne Actinic Keratosis Eczema Melanoma Pigmentation Vitiligo:    1 gambar
    [AKAN DIFILTER] Acne Actinic Keratosis Scars  :    1 gambar
    [AKAN DIFILTER] Acne Scars                    :    1 gambar
    [AKAN

In [11]:
import shutil
import os

# ============================================
# KELAS TARGET — definisi resmi untuk project
# Semua nama akan dinormalisasi ke format ini
# ============================================
TARGET_CLASSES = {
    # Dataset 1 (huruf kecil) → normalize
    "eczema"    : "eczema",
    "psoriasis" : "psoriasis",

    # Dataset 2 (huruf kapital) → normalize
    "Eczema"    : "eczema",
    "Psoriasis" : "psoriasis",

    # Anomali → BUANG (tidak masuk dict ini)
    # "Acne Actinic Keratosis Eczema..." → otomatis terlewat
}

def collect_target_images(raw_base_path, target_classes, dataset_label):
    """
    Kumpulkan semua gambar kelas target dari semua split,
    normalisasi nama kelas, buang kelas lain.

    Return: list of (source_path, normalized_class)
    """
    collected = []
    skipped_classes = set()

    for split in ["train", "valid", "test"]:
        split_path = os.path.join(raw_base_path, split)
        if not os.path.exists(split_path):
            continue

        for cls_folder in os.listdir(split_path):
            cls_path = os.path.join(split_path, cls_folder)
            if not os.path.isdir(cls_path):
                continue

            # Cek apakah kelas ini masuk target
            if cls_folder not in target_classes:
                skipped_classes.add(cls_folder)
                continue

            normalized_cls = target_classes[cls_folder]

            # Kumpulkan semua file gambar
            for fname in os.listdir(cls_path):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    full_path = os.path.join(cls_path, fname)
                    # Rename untuk hindari konflik nama antar dataset
                    new_fname = f"{dataset_label}_{split}_{fname}"
                    collected.append((full_path, normalized_cls, new_fname))

    print(f"\n======== {dataset_label}:")
    print(f"   Kelas diambil : {set(target_classes.values())}")
    print(f"   Kelas dibuang : {skipped_classes}")
    print(f"   Total gambar  : {len(collected)}")
    return collected

# Kumpulkan dari kedua dataset
images_d1 = collect_target_images(
    f"{PROJECT_ROOT}/data/raw/dataset1",
    TARGET_CLASSES,
    "d1"
)

images_d2 = collect_target_images(
    f"{PROJECT_ROOT}/data/raw/dataset2",
    TARGET_CLASSES,
    "d2"
)

# Gabungkan
all_images = images_d1 + images_d2

# Ringkasan per kelas
from collections import Counter
class_counts = Counter(cls for _, cls, _ in all_images)

print(f"\n{'='*45}")
print(f"  TOTAL GABUNGAN")
print(f"{'='*45}")
for cls, count in sorted(class_counts.items()):
    print(f"  {cls:15s}: {count} gambar")
print(f"  {'TOTAL':15s}: {sum(class_counts.values())} gambar")


======== d1:
   Kelas diambil : {'psoriasis', 'eczema'}
   Kelas dibuang : {'scabies'}
   Total gambar  : 865

======== d2:
   Kelas diambil : {'psoriasis', 'eczema'}
   Kelas dibuang : {'Pigmentation Scars', 'Hives', 'Melanoma', 'Scars', 'Vitiligo', 'Tinea corporis', 'Acne Actinic Keratosis Scars', 'Acne', 'Acne Actinic Keratosis', 'Acne Scars', 'Actinic Keratosis', 'Pigmentation', 'Acne Actinic Keratosis Eczema Melanoma Pigmentation Vitiligo'}
   Total gambar  : 294

  TOTAL GABUNGAN
  eczema         : 581 gambar
  psoriasis      : 578 gambar
  TOTAL          : 1159 gambar


In [12]:
from sklearn.model_selection import train_test_split
import random

random.seed(42)  # Reproducibility — wajib dicatat di bab metodologi

# ============================================
# STRATIFIED SPLIT 70 / 15 / 15
# Stratified = proporsi kelas sama di tiap split
# ============================================

# Pisahkan per kelas dulu
eczema_images    = [(p, c, f) for p, c, f in all_images if c == "eczema"]
psoriasis_images = [(p, c, f) for p, c, f in all_images if c == "psoriasis"]

def stratified_split(images, train_ratio=0.70, val_ratio=0.15, seed=42):
    """Split list gambar menjadi train/val/test."""
    random.seed(seed)
    random.shuffle(images)

    n = len(images)
    n_train = int(n * train_ratio)
    n_val   = int(n * val_ratio)

    train = images[:n_train]
    val   = images[n_train:n_train + n_val]
    test  = images[n_train + n_val:]

    return train, val, test

e_train, e_val, e_test = stratified_split(eczema_images)
p_train, p_val, p_test = stratified_split(psoriasis_images)

split_summary = {
    "train" : e_train + p_train,
    "val"   : e_val   + p_val,
    "test"  : e_test  + p_test,
}

print("="*45)
print("  HASIL STRATIFIED SPLIT 70/15/15")
print("="*45)
for split_name, imgs in split_summary.items():
    counts = Counter(cls for _, cls, _ in imgs)
    total  = sum(counts.values())
    print(f"\n  [{split_name.upper()}] total: {total}")
    for cls, cnt in sorted(counts.items()):
        pct = cnt/total*100
        print(f"    {cls:15s}: {cnt:4d} ({pct:.1f}%)")

  HASIL STRATIFIED SPLIT 70/15/15

  [TRAIN] total: 810
    eczema         :  406 (50.1%)
    psoriasis      :  404 (49.9%)

  [VAL] total: 173
    eczema         :   87 (50.3%)
    psoriasis      :   86 (49.7%)

  [TEST] total: 176
    eczema         :   88 (50.0%)
    psoriasis      :   88 (50.0%)


In [13]:
# ============================================
# COPY gambar ke struktur folder final
# data/processed/{split}/{class}/
# ============================================

import shutil

def build_processed_dataset(split_summary, processed_base):
    """Copy gambar ke folder processed dengan struktur bersih."""

    total_copied = 0

    for split_name, images in split_summary.items():
        for src_path, cls, new_fname in images:
            dst_dir  = os.path.join(processed_base, split_name, cls)
            dst_path = os.path.join(dst_dir, new_fname)

            os.makedirs(dst_dir, exist_ok=True)

            # Skip jika sudah ada (idempotent)
            if not os.path.exists(dst_path):
                shutil.copy2(src_path, dst_path)
                total_copied += 1

    return total_copied

print("========== Menyalin gambar ke data/processed...")
print("========== Proses ini butuh beberapa menit...\n")

processed_base = f"{PROJECT_ROOT}/data/processed"
total = build_processed_dataset(split_summary, processed_base)

print(f"========= Selesai! {total} gambar disalin\n")

# Verifikasi final
print("="*45)
print("  VERIFIKASI STRUKTUR PROCESSED")
print("="*45)
for split in ["train", "val", "test"]:
    print(f"\n  [{split.upper()}]")
    for cls in ["eczema", "psoriasis"]:
        path  = os.path.join(processed_base, split, cls)
        count = len(os.listdir(path)) if os.path.exists(path) else 0
        print(f"    {cls:15s}: {count} gambar")

========== Menyalin gambar ke data/processed...
========== Proses ini butuh beberapa menit...

========= Selesai! 1159 gambar disalin

  VERIFIKASI STRUKTUR PROCESSED

  [TRAIN]
    eczema         : 406 gambar
    psoriasis      : 404 gambar

  [VAL]
    eczema         : 87 gambar
    psoriasis      : 86 gambar

  [TEST]
    eczema         : 88 gambar
    psoriasis      : 88 gambar


In [18]:
import os
import shutil

REPO_DIR = "/content/skin-xai-skripsi"

# Pastikan repo masih ada (kalau session baru, clone dulu)
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/srohimatuzz/skin-xai-skripsi.git {REPO_DIR}
    print("✅ Repo di-clone ulang")

# Buat .gitignore dulu — PENTING agar dataset tidak ikut push
gitignore_content = """
# Dataset — terlalu besar untuk GitHub
data/

# Python cache
__pycache__/
*.pyc
.ipynb_checkpoints/

# Model files (opsional, bisa di-comment jika mau push)
outputs/models/*.h5
"""

with open(f"{REPO_DIR}/.gitignore", "w") as f:
    f.write(gitignore_content)

print("✅ .gitignore dibuat")

✅ .gitignore dibuat


In [19]:
# Copy notebook dari Drive ke repo
# (dataset TIDAK dicopy karena ada di .gitignore)

notebook_src = f"{PROJECT_ROOT}/00_Setup_and_EDA.ipynb"
notebook_dst = f"{REPO_DIR}/00_Setup_and_EDA.ipynb"

if os.path.exists(notebook_src):
    shutil.copy2(notebook_src, notebook_dst)
    print(f"✅ Notebook berhasil dicopy ke repo")
else:
    print(f"⚠️ Notebook tidak ditemukan di: {notebook_src}")
    print("Cek nama file notebook kamu di Drive")

⚠️ Notebook tidak ditemukan di: /content/drive/MyDrive/skin-xai-skripsi/00_Setup_and_EDA.ipynb
Cek nama file notebook kamu di Drive


In [20]:
import os

# Cek semua file yang ada di folder project Drive
print("📁 Isi folder skin-xai-skripsi di Drive:\n")

for item in os.listdir(PROJECT_ROOT):
    full_path = os.path.join(PROJECT_ROOT, item)
    tipe = "📁" if os.path.isdir(full_path) else "📄"
    print(f"  {tipe} {item}")

📁 Isi folder skin-xai-skripsi di Drive:

  📁 data
  📁 outputs
  📁 experiments


In [21]:
import os

print("📁 Isi folder skin-xai-skripsi di Drive:\n")
for item in sorted(os.listdir(PROJECT_ROOT)):
    full_path = os.path.join(PROJECT_ROOT, item)
    tipe = "📁" if os.path.isdir(full_path) else "📄"
    print(f"  {tipe} {item}")

📁 Isi folder skin-xai-skripsi di Drive:

  📄 00_Setup_and_EDA.ipynb
  📁 data
  📁 experiments
  📁 outputs
